# smICA local automation server

smICA v2.2.0 provides an optional versioned TCP/JSON interface for scripted operation. The normal graphical application still starts with the server disabled. Infrastructure and scientific commands are registered in the same running application, and scientific work is dispatched to the Dear PyGui thread.

>  **For version v2.2.0 and higher:**

Start the installed application with `run_smICA --automation-server` on Linux or `smICA.bat --automation-server` on Windows. The listener binds only to `127.0.0.1`; the default port is 8765 and can be changed with `--automation-port`. The **About > Automation server** panel reports whether the listener is running and displays the effective local address. Commands are sent by the installed `smica-call` client from another terminal on the same machine.

## Starting and stopping the server

Start the ordinary graphical application:

```bash
run_smICA
```

Start smICA with the automation server immediately enabled:

```bash
run_smICA --automation-server --automation-port 8765
```

Port `0` selects a free port. `--automation-print-startup` prints a one-line JSON startup record containing the effective port:

```bash
run_smICA --automation-server --automation-port 0 --automation-print-startup
```

The same server can be started or stopped after launch from **About → Automation server**. The panel displays status, loopback address, and effective port. It never displays an authentication token.

For a remote-shell workflow, log in to the remote account, start smICA with the option above, and run the client from another shell on the same machine.


## Client

The installer provides both `smica-call` and `smica_call` as text-based Bash scripts on Linux and Batch scripts on Windows. No compiled automation-client launcher is installed. Send a readiness request with:

```bash
smica-call --port 8765 system.health
```

Send a command with a JSON parameter object:

```bash
smica-call --port 8765 command.name '{"parameter": "value"}'
```

Exit status is `0` for a successful server response, `1` for a command error returned by smICA, and `2` for local validation or connection failure.


## Authentication

Authentication is optional for same-account local operation. Supply a token to smICA through `SMICA_AUTOMATION_TOKEN` or through `--automation-token-file`. The client reads the same environment variable or accepts `--token-file`. A token is never required as a command-line literal and is not included in normal logs, status output, or result manifests.

```bash
SMICA_AUTOMATION_TOKEN="$(cat token.txt)" \
  run_smICA --automation-server

smica-call --token-file token.txt system.health
```


## Protocol version 1

Transport is UTF-8 newline-delimited JSON. Each request line contains exactly one object:

```json
{
  "protocol_version": 1,
  "request_id": "example-1",
  "command": "system.health",
  "parameters": {}
}
```

A successful response is:

```json
{
  "protocol_version": 1,
  "request_id": "example-1",
  "ok": true,
  "result": {
    "command_id": "server-generated-id",
    "value": {}
  }
}
```

Errors contain a stable code, concise message, and optional details. Raw tracebacks remain local. Unknown envelope fields, duplicate JSON fields, non-standard constants such as `NaN`, invalid UTF-8, malformed JSON, unsupported versions and oversized messages are rejected.


## Available infrastructure commands

| Command | Purpose |
|---|---|
| `system.health` | Confirm readiness and report the effective listener state. |
| `system.version` | Report application and protocol versions. |
| `system.capabilities` | Return sorted metadata for the current explicit command allowlist. |
| `session.inspect` | Return high-level GUI session state without exposing widget identifiers. |
| `system.shutdown` | Available only when smICA starts with `--automation-allow-shutdown`; intended for controlled tests. |

No command exposes Python evaluation, dynamic import, unrestricted object access, arbitrary widget mutation, or unrestricted file-system operations.


> **Documentation added for v2.2.0**
>

Run `smica-call system.capabilities '{}'` against the active application to obtain the authoritative, version-specific command list. The result includes every public command name, description and GUI-thread requirement. Human-readable command groups and examples are maintained in [Automation commands](Automation_commands.ipynb), but scripts should use the live capability response when validating availability.

## Threading and lifecycle

The socket listener runs in a background thread. It validates and authenticates requests, then places them in a bounded queue. The controlled Dear PyGui render loop processes a limited number per frame on the GUI thread. The transport thread never calls Dear PyGui.

Stopping the server closes the listener while leaving the application and dispatcher available for a later manual restart. Closing smICA stops the listener, cancels queued requests, stops the documentation server, and destroys the Dear PyGui context. Server startup and shutdown are idempotent.


## Limits and error handling

The listener binds only to `127.0.0.1`. Configurable limits cover request bytes, socket read time, command completion time, queue capacity and commands processed per GUI frame. The principal infrastructure error codes are `INVALID_ENCODING`, `MALFORMED_JSON`, `INCOMPLETE_REQUEST`, `REQUEST_TOO_LARGE`, `INVALID_ENVELOPE`, `UNSUPPORTED_PROTOCOL_VERSION`, `AUTHENTICATION_FAILED`, `UNKNOWN_COMMAND`, `INVALID_PARAMETERS`, `QUEUE_FULL`, `COMMAND_TIMEOUT`, `CANCELLED`, `DISPATCHER_CLOSED`, `OPERATION_UNAVAILABLE` and `INTERNAL_ERROR`.

Native GUI batch callbacks report their progress in the application. The external workflow runner records completed batches and can resume without recalculating outputs already accepted as complete. The TCP interface does not currently expose general-purpose job status or mid-operation cancellation commands.

> **Documentation added for v2.2.0**
>

An internal command failure names the operation that failed:

```json
{
  "protocol_version": 1,
  "request_id": "cli-example",
  "ok": false,
  "error": {
    "code": "INTERNAL_ERROR",
    "message": "The command 'extract.apply_all' failed inside smICA. See the local log for details.",
    "details": {"command": "extract.apply_all"}
  }
}
```

The response is intentionally compact. The complete traceback is written to the local smICA log and is not transmitted to the client.

## Verification

The non-interactive suite covers protocol encoding and validation, duplicate command rejection, parameter validation, queue saturation, cancellation, local error redaction, execution-thread identity, fragmented requests, multiple requests per connection, malformed traffic, request limits, authentication, concurrent clients, operation timeout, client disconnect, graceful listener shutdown, client exit behaviour and real Dear PyGui panel creation.

The server starts disabled when no automation option is supplied. Linux and Windows launchers forward optional arguments. Every Dear PyGui item added by the automation panel has an explicit tag and is registered in `globalITEMS.items` so ordinary mode switching remains safe.